# The Prism Recipe

**13x faster to baseline quality. Best val loss baseline never reaches. Zero overfitting.**

Three ingredients:
1. **EigenTransfer** (align 0.75) — inject pretrained singular vector directions
2. **Spectral Imprint** (8 DCT coefficients) — shape the singular value distribution
3. **Mod Wheel** (strength 0.01, decay 0.9999) — continuous anti-overfitting modulation

Evaluated on held-out test data with strict partitioning.

---
*[Sean McDonald](https://x.com/seanmcdonaldxyz) · [github.com/timepointai/nanogpt-prism-shakespeare](https://github.com/timepointai/nanogpt-prism-shakespeare)*

In [ ]:
# ═══════════════════════════════════════════
# STEP 1: Setup
# ═══════════════════════════════════════════
import os, subprocess, shutil, numpy as np

os.chdir('/content')
if os.path.exists('/content/nanogpt-prism'):
    shutil.rmtree('/content/nanogpt-prism')
subprocess.run(['git', 'clone',
    'https://github.com/timepointai/nanogpt-prism-shakespeare.git',
    '/content/nanogpt-prism'], check=True)
os.chdir('/content/nanogpt-prism')
subprocess.run(['pip', 'install', '-q', 'transformers', 'tiktoken', 'datasets'])

import torch
gpu = torch.cuda.get_device_name(0)
print(f'GPU: {gpu}')

# Prepare Shakespeare
subprocess.run(['python', 'data/shakespeare_char/prepare.py'],
               capture_output=True)

# Strict partition: train (80%) / teacher-val (20%) / test (original val)
train_all = np.array(np.memmap('data/shakespeare_char/train.bin',
                                dtype=np.uint16, mode='r'))
test_data = np.array(np.memmap('data/shakespeare_char/val.bin',
                                dtype=np.uint16, mode='r'))
split = int(len(train_all) * 0.80)

for name, val in [('shakespeare_eval', test_data),
                   ('shakespeare_teacher', train_all[split:])]:
    d = f'data/{name}'
    os.makedirs(d, exist_ok=True)
    train_all[:split].astype(np.uint16).tofile(f'{d}/train.bin')
    val.astype(np.uint16).tofile(f'{d}/val.bin')
    shutil.copy('data/shakespeare_char/meta.pkl', f'{d}/meta.pkl')

print(f'Train: {split:,} tokens | Test: {len(test_data):,} tokens (held out)')
print('Ready.')

In [ ]:
# ═══════════════════════════════════════════
# STEP 2: Train teacher + extract fingerprint
# ═══════════════════════════════════════════
import subprocess, os, re
os.chdir('/content/nanogpt-prism')

if os.path.exists('.prism_cache/teacher/directions.pt'):
    print('Teacher fingerprint cached.')
else:
    print('Training teacher (2000 steps)...')
    r = subprocess.run([
        'python', 'train.py', 'config/train_shakespeare_char.py',
        '--dataset=shakespeare_teacher', '--max_iters=2000',
        '--eval_interval=2000', '--eval_iters=50', '--log_interval=2000',
        '--out_dir=out-recipe-teacher', '--always_save_checkpoint=True',
        '--compile=False', '--prism_init=False', '--wandb_log=False',
    ], capture_output=True, text=True, timeout=600)
    for line in r.stdout.split('\n'):
        m = re.search(r'step \d+: .+ val loss ([\d.]+)', line)
        if m: print(f'  Teacher val loss: {m.group(1)}')

    print('Extracting spectral fingerprint...')
    subprocess.run([
        'python', 'prism_extract.py',
        '--ckpt', 'out-recipe-teacher/ckpt.pt',
        '--out', '.prism_cache/teacher',
    ], capture_output=True, timeout=120)
    print('Done.')

In [ ]:
# ═══════════════════════════════════════════
# STEP 3: Race — Baseline vs Prism Recipe
# ═══════════════════════════════════════════
import subprocess, time, re, json, os
os.chdir('/content/nanogpt-prism')

STEPS = 5000

runs = [
    ('Baseline', [
        '--prism_init=False',
    ]),
    ('Prism Recipe', [
        '--prism_init=True',
        '--prism_align=0.75',
        '--prism_spectra=.prism_cache/teacher/spectra.json',
        '--prism_directions=.prism_cache/teacher/directions.pt',
        '--learning_rate=5e-4',
        '--warmup_iters=50',
        '--prism_mod=0.01',
        '--prism_mod_decay=0.9999',
    ]),
]

results = {}
for display_name, extra in runs:
    tag = display_name.lower().replace(' ', '_')
    log_path = f'/content/recipe_{tag}.txt'

    print(f'\n  {"═"*50}')
    print(f'  {display_name}')
    print(f'  {"═"*50}')

    t0 = time.time()
    r = subprocess.run(
        ['python', 'train.py', 'config/train_shakespeare_char.py',
         '--dataset=shakespeare_eval',
         f'--max_iters={STEPS}', '--eval_interval=100',
         '--eval_iters=50', '--log_interval=250',
         f'--out_dir=out-recipe-{tag}',
         '--wandb_log=False', '--compile=False'] + extra,
        capture_output=True, text=True, timeout=1200
    )
    wall = time.time() - t0

    with open(log_path, 'w') as f:
        f.write(r.stdout)
    if r.returncode != 0:
        print(f'  ERROR: {r.stderr[-300:]}')
        continue

    val = {}
    for line in r.stdout.split('\n'):
        m = re.search(r'step (\d+): train loss ([\d.]+), val loss ([\d.]+)', line)
        if m: val[int(m.group(1))] = float(m.group(3))

    best = min(val.values()) if val else 999
    best_s = min(val, key=val.get) if val else 0
    at_5k = val.get(STEPS, val.get(max(val.keys()), 0)) if val else 0

    results[display_name] = {'val': val, 'best': best, 'best_step': best_s,
                             'at_5000': at_5k, 'wall': wall}
    print(f'  Best: {best:.4f} @ step {best_s}')
    print(f'  @5000: {at_5k:.4f}')
    print(f'  Time: {wall:.0f}s')

# Save
with open('/content/recipe_results.json', 'w') as f:
    out = {}
    for k, v in results.items():
        out[k] = {kk: vv for kk, vv in v.items() if kk != 'val'}
        out[k]['val'] = {str(s): l for s, l in v['val'].items()}
    json.dump(out, f, indent=2)
print('\nSaved.')

In [ ]:
# ═══════════════════════════════════════════
# STEP 4: The Verdict
# ═══════════════════════════════════════════
import json

data = json.load(open('/content/recipe_results.json'))
for k in data:
    data[k]['val'] = {int(s): v for s, v in data[k]['val'].items()}

b = data['Baseline']
p = data['Prism Recipe']

target = b['best']
target_step = b['best_step']

prism_hit = None
for s in sorted(p['val'].keys()):
    if p['val'][s] <= target:
        prism_hit = s
        break
speedup = target_step / prism_hit if prism_hit else 0

print()
print('  ┌───────────────────────────────────────────────────┐')
print('  │           THE PRISM RECIPE — RESULTS              │')
print('  ├───────────────────────────────────────────────────┤')
print(f'  │  GPU: {data["Baseline"]["val"] and "A100" or "?":>43s} │')
print(f'  │  Eval: held-out test set, strict partition       │')
print('  ├───────────────┬──────────────┬────────────────────┤')
print('  │               │   Baseline   │   Prism Recipe     │')
print('  ├───────────────┼──────────────┼────────────────────┤')
print(f'  │ Best val loss │   {target:>8.4f}   │   {p["best"]:>8.4f}           │')
print(f'  │ Best @ step   │   {target_step:>8d}   │   {p["best_step"]:>8d}           │')
print(f'  │ Val @ 5000    │   {b["at_5000"]:>8.4f}   │   {p["at_5000"]:>8.4f}           │')
print(f'  │ Overfitting   │   {"YES":>8s}   │   {"NO":>8s}           │')
print('  ├───────────────┴──────────────┴────────────────────┤')
if prism_hit:
    print(f'  │  Prism reaches baseline quality at step {prism_hit:>5d}      │')
    print(f'  │  Baseline reaches it at step {target_step:>5d}                │')
    print(f'  │                                                   │')
    print(f'  │  >>> {speedup:.0f}x FASTER <<<{" ":>28s}│')
print('  │                                                   │')
print(f'  │  Prism best ({p["best"]:.4f}) is LOWER than baseline  │')
print(f'  │  ever achieves at any point in 5000 steps.        │')
print('  └───────────────────────────────────────────────────┘')

# Convergence curve
print(f'\n  {"Step":>6s}  {"Baseline":>10s}  {"Prism":>10s}  {"Winner":>8s}')
print(f'  {"-"*38}')
all_steps = sorted(set(b['val'].keys()) & set(p['val'].keys()))
for step in all_steps:
    if step % 500 == 0 or step <= 500 and step % 100 == 0:
        bv = b['val'][step]
        pv = p['val'][step]
        w = 'PRISM' if pv < bv else 'BASE'
        print(f'  {step:>6d}  {bv:>10.4f}  {pv:>10.4f}  {w:>8s}')

In [ ]:
# ═══════════════════════════════════════════
# STEP 5: Download
# ═══════════════════════════════════════════
from google.colab import files
for f in ['/content/recipe_results.json',
          '/content/recipe_baseline.txt',
          '/content/recipe_prism_recipe.txt']:
    try: files.download(f)
    except: pass